# GAN Assignment — Variants 1 (Light + PRO + Ultra PRO + Kaggle)

| Задание | Описание |
|---------|----------|
| **Light** | DCGAN — генерация рукописных цифр (MNIST) |
| **PRO** | CGAN — Fashion MNIST, лейбл = картинка цифры 28×28 из MNIST |
| **Ultra PRO** | Рисуем цифру прямо в ноутбуке → генерируем одежду |
| **Kaggle** | DCGAN на Cats vs Dogs (64×64 RGB) |

> **Google Colab**: `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ── Общие зависимости ──────────────────────────────────────────────────────
import torch, torch.nn as nn, torch.optim as optim
import torchvision
import torchvision.transforms as T
from torchvision.utils import make_grid, save_image
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader
import os, io, base64
from PIL import Image, ImageOps

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

def show(tensor, nrow=8, title='', figsize=(12,4)):
    grid = make_grid(tensor.clamp(-1,1), nrow=nrow, normalize=True, value_range=(-1,1))
    plt.figure(figsize=figsize)
    plt.title(title, fontsize=13)
    plt.imshow(grid.permute(1,2,0).cpu().numpy())
    plt.axis('off'); plt.tight_layout(); plt.show()

def weights_init(m):
    classname = m.__class__.__name__
    if 'Conv' in classname:
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif 'BatchNorm' in classname:
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)

---
## 🟢 LIGHT — DCGAN: генерация цифр MNIST

In [ ]:
Z_DIM = 100; BATCH = 128; LR = 2e-4; EPOCHS = 30

tf_mnist = T.Compose([T.ToTensor(), T.Normalize([0.5], [0.5])])
mnist_ds = torchvision.datasets.MNIST('./data', train=True, download=True, transform=tf_mnist)
mnist_dl = DataLoader(mnist_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)

In [ ]:
class Generator(nn.Module):
    """z(100) → (1,28,28)"""
    def __init__(self, z_dim=100):
        super().__init__()
        self.fc = nn.Linear(z_dim, 256*7*7)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(True),  # 14
            nn.ConvTranspose2d(128,  64, 4, 2, 1), nn.BatchNorm2d(64),  nn.ReLU(True),  # 28
            nn.Conv2d(64, 1, 3, 1, 1), nn.Tanh(),
        )
    def forward(self, z):
        return self.net(self.fc(z).view(-1, 256, 7, 7))

class Discriminator(nn.Module):
    """(1,28,28) → logit"""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,   64, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, True),
            nn.Conv2d(128,256, 3, 1, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2, True),
            nn.Flatten(), nn.Linear(256*7*7, 1),
        )
    def forward(self, x): return self.net(x)

G = Generator(Z_DIM).to(DEVICE).apply(weights_init)
D = Discriminator().to(DEVICE).apply(weights_init)
print(f'G: {sum(p.numel() for p in G.parameters()):,}  D: {sum(p.numel() for p in D.parameters()):,}')

In [ ]:
crit  = nn.BCEWithLogitsLoss()
opt_G = optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))
fixed_z = torch.randn(64, Z_DIM, device=DEVICE)
hist = {'d':[], 'g':[]}

for epoch in range(1, EPOCHS+1):
    de, ge = 0., 0.
    for real, _ in mnist_dl:
        real = real.to(DEVICE); bs = real.size(0)
        ones  = torch.ones(bs,  1, device=DEVICE)
        zeros = torch.zeros(bs, 1, device=DEVICE)

        fake   = G(torch.randn(bs, Z_DIM, device=DEVICE)).detach()
        loss_D = crit(D(real), ones) + crit(D(fake), zeros)
        opt_D.zero_grad(); loss_D.backward(); opt_D.step()

        fake   = G(torch.randn(bs, Z_DIM, device=DEVICE))
        loss_G = crit(D(fake), ones)
        opt_G.zero_grad(); loss_G.backward(); opt_G.step()
        de += loss_D.item(); ge += loss_G.item()

    n = len(mnist_dl)
    hist['d'].append(de/n); hist['g'].append(ge/n)
    if epoch % 5 == 0 or epoch == 1:
        print(f'[{epoch:3d}/{EPOCHS}] D={de/n:.3f} G={ge/n:.3f}')
        with torch.no_grad(): show(G(fixed_z), title=f'DCGAN epoch {epoch}')

plt.figure(figsize=(8,3))
plt.plot(hist['d'], label='D'); plt.plot(hist['g'], label='G')
plt.title('DCGAN MNIST loss'); plt.legend(); plt.show()

---
## 🔵 PRO — CGAN: Fashion MNIST с MNIST-лейблом (28×28)

In [ ]:
Z_DIM_C = 128; EMB_DIM = 256; BATCH_C = 64; LR_C = 1e-4; EPOCHS_C = 50
FASHION_CLASSES = ['T-shirt','Trouser','Pullover','Dress','Coat',
                   'Sandal','Shirt','Sneaker','Bag','Ankle boot']

tf = T.Compose([T.ToTensor(), T.Normalize([0.5],[0.5])])
fashion_ds = torchvision.datasets.FashionMNIST('./data', train=True, download=True, transform=tf)
fashion_dl = DataLoader(fashion_ds, batch_size=BATCH_C, shuffle=True,
                        num_workers=2, pin_memory=True, drop_last=True)
mnist_all  = torchvision.datasets.MNIST('./data', train=True, download=True, transform=tf)

label_imgs = {}
for img, lbl in mnist_all:
    if lbl not in label_imgs: label_imgs[lbl] = img
    if len(label_imgs) == 10: break

COND_BANK = torch.stack([label_imgs[i] for i in range(10)]).unsqueeze(1).to(DEVICE)  # (10,1,28,28)
show(COND_BANK, nrow=10, title='MNIST-цифры как лейблы')

In [ ]:
class LabelEncoder(nn.Module):
    """(1,28,28) → (EMB_DIM,)"""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, 2, 1), nn.LeakyReLU(0.2, True),   # 14
            nn.Conv2d(32,64, 3, 2, 1), nn.LeakyReLU(0.2, True),   #  7
            nn.Conv2d(64,128,3, 2, 1), nn.LeakyReLU(0.2, True),   #  4
            nn.Flatten(), nn.Linear(128*4*4, EMB_DIM), nn.LayerNorm(EMB_DIM),
        )
    def forward(self, x): return self.net(x)

class CGANGenerator(nn.Module):
    """z + emb → (1,28,28)"""
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(Z_DIM_C + EMB_DIM, 512*7*7),
            nn.BatchNorm1d(512*7*7), nn.ReLU(True),
        )
        self.up = nn.Sequential(
            nn.ConvTranspose2d(512,256,4,2,1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256,128,4,2,1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.Conv2d(128, 64,3,1,1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64,   1,3,1,1), nn.Tanh(),
        )
    def forward(self, z, e):
        return self.up(self.fc(torch.cat([z,e],1)).view(-1,512,7,7))

class CGANDiscriminator(nn.Module):
    """img + emb → logit"""
    def __init__(self):
        super().__init__()
        self.img = nn.Sequential(nn.Conv2d(1,64,4,2,1), nn.LeakyReLU(0.2,True))
        self.lbl = nn.Linear(EMB_DIM, 14*14)
        self.out = nn.Sequential(
            nn.Conv2d(65,128,4,2,1), nn.LeakyReLU(0.2,True),
            nn.Conv2d(128,256,3,1,1), nn.LeakyReLU(0.2,True),
            nn.Flatten(), nn.Linear(256*7*7, 1),
        )
    def forward(self, img, e):
        return self.out(torch.cat([self.img(img), self.lbl(e).view(-1,1,14,14)], 1))

enc = LabelEncoder().to(DEVICE).apply(weights_init)
cG  = CGANGenerator().to(DEVICE).apply(weights_init)
cD  = CGANDiscriminator().to(DEVICE).apply(weights_init)

In [ ]:
crit_c  = nn.BCEWithLogitsLoss()
opt_cG  = optim.Adam(list(cG.parameters()) + list(enc.parameters()), lr=LR_C, betas=(0.5,0.999))
opt_cD  = optim.Adam(cD.parameters(), lr=LR_C, betas=(0.5,0.999))
sch_G   = optim.lr_scheduler.CosineAnnealingLR(opt_cG, T_max=EPOCHS_C)
sch_D   = optim.lr_scheduler.CosineAnnealingLR(opt_cD, T_max=EPOCHS_C)

fixed_z_c   = torch.randn(60, Z_DIM_C, device=DEVICE)
fixed_cls   = torch.arange(10, device=DEVICE).repeat_interleave(6)
fixed_conds = COND_BANK[fixed_cls]  # (60,1,28,28)
hist_c = {'d':[], 'g':[]}

for epoch in range(1, EPOCHS_C+1):
    de, ge = 0., 0.
    for real_img, real_cls in fashion_dl:
        real_img = real_img.to(DEVICE); real_cls = real_cls.to(DEVICE)
        bs = real_img.size(0)
        conds    = COND_BANK[real_cls]
        emb      = enc(conds)
        ones     = torch.ones(bs,  1, device=DEVICE)
        zeros    = torch.zeros(bs, 1, device=DEVICE)
        wrong    = (real_cls + torch.randint(1,10,(bs,),device=DEVICE)) % 10

        fake  = cG(torch.randn(bs,Z_DIM_C,device=DEVICE), emb).detach()
        loss_D = (crit_c(cD(real_img, emb),  ones)  +
                  crit_c(cD(fake,     emb),  zeros) +
                  crit_c(cD(real_img, enc(COND_BANK[wrong])), zeros)) / 3
        opt_cD.zero_grad(); loss_D.backward(); opt_cD.step()

        emb  = enc(conds)  # пересчёт для графа
        fake = cG(torch.randn(bs,Z_DIM_C,device=DEVICE), emb)
        loss_G = crit_c(cD(fake, emb), ones)
        opt_cG.zero_grad(); loss_G.backward(); opt_cG.step()
        de += loss_D.item(); ge += loss_G.item()

    sch_G.step(); sch_D.step()
    n = len(fashion_dl); hist_c['d'].append(de/n); hist_c['g'].append(ge/n)
    if epoch % 10 == 0 or epoch == 1:
        print(f'[{epoch:3d}/{EPOCHS_C}] D={de/n:.3f} G={ge/n:.3f}')
        with torch.no_grad():
            show(cG(fixed_z_c, enc(fixed_conds)), nrow=10,
                 title=f'CGAN epoch {epoch} — строки: цифра 0→9 = класс одежды')

# Итоговая таблица
with torch.no_grad(): final = cG(fixed_z_c, enc(fixed_conds))
fig, axes = plt.subplots(2, 10, figsize=(16,4))
for i in range(10):
    axes[0,i].imshow(COND_BANK[i].squeeze().cpu(), cmap='gray')
    axes[0,i].set_title(str(i), fontsize=10); axes[0,i].axis('off')
    axes[1,i].imshow((final[i*6].squeeze().cpu()+1)/2, cmap='gray')
    axes[1,i].set_title(FASHION_CLASSES[i], fontsize=7); axes[1,i].axis('off')
plt.suptitle('CGAN: MNIST digit → Fashion MNIST'); plt.tight_layout(); plt.show()

---
## 🔴 Ultra PRO — Рисуем цифру прямо в ноутбуке

Запустите ячейку → появится холст. Нарисуйте цифру мышью → нажмите **"Использовать"**.

In [ ]:
# ── Интерактивный canvas для рисования прямо в Colab ──────────────────────
import google.colab.output
from IPython.display import display, HTML

drawn_img_t = None  # сюда запишем тензор после нажатия кнопки

def _on_canvas_submit(data_url):
    global drawn_img_t
    header, encoded = data_url.split(',', 1)
    raw = base64.b64decode(encoded)
    img = Image.open(io.BytesIO(raw)).convert('L')
    img = ImageOps.autocontrast(img)
    img = img.resize((28, 28), Image.LANCZOS)
    arr = np.array(img, dtype=np.float32)
    if arr.mean() > 127: arr = 255 - arr          # инвертируем, если фон светлый
    drawn_img_t = torch.tensor(arr / 127.5 - 1.0).unsqueeze(0).unsqueeze(0).to(DEVICE)

    # Показываем результат
    plt.figure(figsize=(3,3))
    plt.imshow(arr, cmap='gray')
    plt.title('Ваша цифра (28×28)', fontsize=12); plt.axis('off'); plt.show()
    print('✓ Готово! Теперь запустите следующую ячейку.')

google.colab.output.register_callback('notebook.canvas_submit', _on_canvas_submit)

CANVAS_HTML = """
<style>
  #draw-container { display:flex; flex-direction:column; align-items:flex-start; gap:8px; }
  #draw-canvas   { border:2px solid #555; border-radius:6px; cursor:crosshair;
                   background:#000; touch-action:none; }
  .draw-btn { padding:6px 18px; border:none; border-radius:4px; font-size:14px;
              cursor:pointer; font-weight:600; }
  #btn-clear { background:#444; color:#fff; }
  #btn-use   { background:#1a73e8; color:#fff; }
  #btn-use:hover { background:#1558b0; }
</style>
<div id="draw-container">
  <span style="font-size:13px;color:#555">Рисуйте белым на чёрном фоне. Толщина кисти — 18px.</span>
  <canvas id="draw-canvas" width="280" height="280"></canvas>
  <div style="display:flex;gap:10px">
    <button class="draw-btn" id="btn-clear">🗑 Очистить</button>
    <button class="draw-btn" id="btn-use">✅ Использовать</button>
  </div>
</div>
<script>
(function(){
  const canvas = document.getElementById('draw-canvas');
  const ctx    = canvas.getContext('2d');
  let drawing  = false;

  function clear(){
    ctx.fillStyle = '#000';
    ctx.fillRect(0, 0, canvas.width, canvas.height);
  }
  clear();

  ctx.strokeStyle = '#fff';
  ctx.lineWidth   = 18;
  ctx.lineCap     = 'round';
  ctx.lineJoin    = 'round';

  function pos(e){
    const r = canvas.getBoundingClientRect();
    if (e.touches) {
      return { x: e.touches[0].clientX - r.left, y: e.touches[0].clientY - r.top };
    }
    return { x: e.clientX - r.left, y: e.clientY - r.top };
  }

  canvas.addEventListener('mousedown',  e => { drawing=true; ctx.beginPath(); const p=pos(e); ctx.moveTo(p.x,p.y); });
  canvas.addEventListener('mousemove',  e => { if(!drawing) return; const p=pos(e); ctx.lineTo(p.x,p.y); ctx.stroke(); ctx.beginPath(); ctx.moveTo(p.x,p.y); });
  canvas.addEventListener('mouseup',    () => drawing=false);
  canvas.addEventListener('mouseleave', () => drawing=false);
  canvas.addEventListener('touchstart', e => { e.preventDefault(); drawing=true; ctx.beginPath(); const p=pos(e); ctx.moveTo(p.x,p.y); }, {passive:false});
  canvas.addEventListener('touchmove',  e => { e.preventDefault(); if(!drawing) return; const p=pos(e); ctx.lineTo(p.x,p.y); ctx.stroke(); ctx.beginPath(); ctx.moveTo(p.x,p.y); }, {passive:false});
  canvas.addEventListener('touchend',   () => drawing=false);

  document.getElementById('btn-clear').onclick = clear;
  document.getElementById('btn-use').onclick   = function(){
    const data = canvas.toDataURL('image/png');
    google.colab.kernel.invokeFunction('notebook.canvas_submit', [data], {});
  };
})();
</script>
"""
display(HTML(CANVAS_HTML))

In [ ]:
# ── Генерация по нарисованной цифре ───────────────────────────────────────
assert drawn_img_t is not None, 'Сначала нарисуйте цифру и нажмите "Использовать"!'

N = 32
with torch.no_grad():
    draw_emb  = enc(drawn_img_t.expand(N, -1, -1, -1))
    z_draw    = torch.randn(N, Z_DIM_C, device=DEVICE)
    generated = cG(z_draw, draw_emb)

show(generated, nrow=8, title='Генерация по нарисованной цифре')

# Лучший по оценке дискриминатора
with torch.no_grad():
    scores = cD(generated, draw_emb).squeeze()
best = generated[scores.argmax()].squeeze().cpu().numpy()
plt.figure(figsize=(3,3))
plt.imshow((best+1)/2, cmap='gray'); plt.title('Лучший сэмпл (D-score)'); plt.axis('off'); plt.show()

In [ ]:
# ── Интерполяция: нарисованная цифра ↔ любой класс из банка ───────────────
TARGET_CLASS = 5  # ← измените (0–9)
STEPS = 10

with torch.no_grad():
    emb_a = enc(drawn_img_t)
    emb_b = enc(COND_BANK[[TARGET_CLASS]])
    t     = torch.linspace(0, 1, STEPS, device=DEVICE).unsqueeze(1)
    embs  = (1-t)*emb_a + t*emb_b
    z_int = torch.randn(STEPS, Z_DIM_C, device=DEVICE)
    interp= cG(z_int, embs)

show(interp, nrow=STEPS, title=f'Интерполяция: ваша цифра → класс {TARGET_CLASS} ({FASHION_CLASSES[TARGET_CLASS]})')

---
## 🟡 Kaggle — DCGAN: Cats vs Dogs (64×64 RGB)

### Шаг 1 — Получить API-ключ Kaggle
1. kaggle.com → Your Profile → Settings → **Create New Token** → скачает `kaggle.json`
2. Запустите ячейку ниже → загрузите файл

In [ ]:
# ── Установка Kaggle API + загрузка ключа ──────────────────────────────────
!pip install -q kaggle
from google.colab import files
print('Загрузите kaggle.json...')
uploaded_k = files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
print('✓ Ключ сохранён')

In [ ]:
# ── Скачиваем датасет ──────────────────────────────────────────────────────
# Dogs vs Cats Redux (маленький вариант — только train.zip ~543 MB)
!kaggle competitions download -c dogs-vs-cats-redux-kernels-edition -p ./data/catsdogs --quiet
!unzip -q ./data/catsdogs/train.zip -d ./data/catsdogs/
print('Файлов:', len(os.listdir('./data/catsdogs/train')))

In [ ]:
# ── Датасет + DataLoader ───────────────────────────────────────────────────
from torchvision.datasets import ImageFolder
from torchvision import datasets
import shutil, pathlib

IMG_SIZE = 64
BATCH_K  = 64
Z_DIM_K  = 128
LR_K     = 2e-4
EPOCHS_K = 60

# ImageFolder ожидает подпапки — переоргаизуем структуру
SRC  = pathlib.Path('./data/catsdogs/train')
DST  = pathlib.Path('./data/catsdogs/sorted')
(DST/'cat').mkdir(parents=True, exist_ok=True)
(DST/'dog').mkdir(parents=True, exist_ok=True)

for f in SRC.glob('*.jpg'):
    label = 'cat' if f.name.startswith('cat') else 'dog'
    target = DST / label / f.name
    if not target.exists():
        shutil.copy(f, target)
print(f"cats: {len(list((DST/'cat').glob('*')))}  dogs: {len(list((DST/'dog').glob('*')))}")

tf_k = T.Compose([
    T.Resize(IMG_SIZE + 4),
    T.RandomCrop(IMG_SIZE),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5]),
])
catdog_ds = ImageFolder(str(DST), transform=tf_k)
catdog_dl = DataLoader(catdog_ds, batch_size=BATCH_K, shuffle=True,
                       num_workers=2, pin_memory=True, drop_last=True)
print(f'Dataset size: {len(catdog_ds)}  Batches/epoch: {len(catdog_dl)}')

In [ ]:
# ── Архитектура DCGAN 64×64 RGB ────────────────────────────────────────────
NGF = 64  # base feature maps
NDF = 64

class GeneratorK(nn.Module):
    """z(128) → (3,64,64)"""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # 1×1 → 4×4
            nn.ConvTranspose2d(Z_DIM_K, NGF*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(NGF*8), nn.ReLU(True),
            # 4 → 8
            nn.ConvTranspose2d(NGF*8, NGF*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF*4), nn.ReLU(True),
            # 8 → 16
            nn.ConvTranspose2d(NGF*4, NGF*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF*2), nn.ReLU(True),
            # 16 → 32
            nn.ConvTranspose2d(NGF*2, NGF,   4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF),   nn.ReLU(True),
            # 32 → 64
            nn.ConvTranspose2d(NGF,   3,     4, 2, 1, bias=False),
            nn.Tanh(),
        )
    def forward(self, z): return self.net(z.view(-1, Z_DIM_K, 1, 1))


class DiscriminatorK(nn.Module):
    """(3,64,64) → logit"""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # 64 → 32
            nn.Conv2d(3,     NDF,   4, 2, 1, bias=False), nn.LeakyReLU(0.2, True),
            # 32 → 16
            nn.Conv2d(NDF,   NDF*2, 4, 2, 1, bias=False), nn.BatchNorm2d(NDF*2), nn.LeakyReLU(0.2, True),
            # 16 → 8
            nn.Conv2d(NDF*2, NDF*4, 4, 2, 1, bias=False), nn.BatchNorm2d(NDF*4), nn.LeakyReLU(0.2, True),
            # 8 → 4
            nn.Conv2d(NDF*4, NDF*8, 4, 2, 1, bias=False), nn.BatchNorm2d(NDF*8), nn.LeakyReLU(0.2, True),
            # 4 → 1
            nn.Conv2d(NDF*8, 1,     4, 1, 0, bias=False),
            nn.Flatten(),
        )
    def forward(self, x): return self.net(x)


GK = GeneratorK().to(DEVICE).apply(weights_init)
DK = DiscriminatorK().to(DEVICE).apply(weights_init)
print(f'G: {sum(p.numel() for p in GK.parameters()):,}  D: {sum(p.numel() for p in DK.parameters()):,}')

# Проверка размерностей
with torch.no_grad():
    _z = torch.randn(4, Z_DIM_K, device=DEVICE)
    _x = GK(_z)
    print(f'Generator output: {_x.shape}  →  {DK(_x).shape}')

In [ ]:
# ── Обучение DCGAN на Cats vs Dogs ─────────────────────────────────────────
crit_k  = nn.BCEWithLogitsLoss()
opt_GK  = optim.Adam(GK.parameters(), lr=LR_K,     betas=(0.5, 0.999))
opt_DK  = optim.Adam(DK.parameters(), lr=LR_K*0.4, betas=(0.5, 0.999))  # D чуть медленнее

fixed_zk = torch.randn(64, Z_DIM_K, device=DEVICE)
hist_k   = {'d':[], 'g':[]}

best_g_loss = float('inf')

for epoch in range(1, EPOCHS_K+1):
    de, ge = 0., 0.
    for real, _ in catdog_dl:
        real = real.to(DEVICE); bs = real.size(0)
        # Метки с label smoothing для стабильности
        ones  = torch.empty(bs, 1, device=DEVICE).uniform_(0.85, 1.0)
        zeros = torch.zeros(bs, 1, device=DEVICE)

        # ─ D ─
        fake   = GK(torch.randn(bs, Z_DIM_K, device=DEVICE)).detach()
        loss_D = crit_k(DK(real), ones) + crit_k(DK(fake), zeros)
        opt_DK.zero_grad(); loss_D.backward(); opt_DK.step()

        # ─ G (2 шага на 1 шаг D для ускорения обучения G) ─
        for _ in range(2):
            fake   = GK(torch.randn(bs, Z_DIM_K, device=DEVICE))
            loss_G = crit_k(DK(fake), torch.ones(bs, 1, device=DEVICE))
            opt_GK.zero_grad(); loss_G.backward(); opt_GK.step()
        de += loss_D.item(); ge += loss_G.item()

    n = len(catdog_dl); hist_k['d'].append(de/n); hist_k['g'].append(ge/n)

    if epoch % 10 == 0 or epoch == 1:
        print(f'[{epoch:3d}/{EPOCHS_K}] D={de/n:.3f} G={ge/n:.3f}')
        with torch.no_grad(): show(GK(fixed_zk), nrow=8, title=f'Cats&Dogs epoch {epoch}')

    # Сохраняем лучший чекпойнт
    if ge/n < best_g_loss:
        best_g_loss = ge/n
        torch.save(GK.state_dict(), './data/catsdogs/best_G.pth')

print(f'Лучший G-loss: {best_g_loss:.4f}')

In [ ]:
# ── Финальные результаты Kaggle ────────────────────────────────────────────
GK.load_state_dict(torch.load('./data/catsdogs/best_G.pth', map_location=DEVICE))

with torch.no_grad():
    samples_k = GK(torch.randn(64, Z_DIM_K, device=DEVICE))

show(samples_k, nrow=8, figsize=(14,8), title='DCGAN Cats & Dogs — лучшие сэмплы')

# Потери
plt.figure(figsize=(9,4))
plt.plot(hist_k['d'], label='Discriminator')
plt.plot(hist_k['g'], label='Generator')
plt.title('DCGAN Cats vs Dogs — Loss'); plt.legend(); plt.show()

# Сохраняем топ-16 в файл
with torch.no_grad():
    top16 = GK(torch.randn(16, Z_DIM_K, device=DEVICE))
save_image(top16, './data/catsdogs/best_samples.png', nrow=4, normalize=True, value_range=(-1,1))
print('Сохранено: ./data/catsdogs/best_samples.png')

In [ ]:
# ── Скачать лучшие фото ────────────────────────────────────────────────────
from google.colab import files
files.download('./data/catsdogs/best_samples.png')

---
## Описание архитектур

### DCGAN — MNIST (Light)
| Блок | Детали |
|------|--------|
| G | `Linear(100→256·7·7)` → `ConvT(128)` → `ConvT(64)` → `Conv(1)` + Tanh |
| D | `Conv(64)` → `Conv(128)` → `Conv(256)` → `Linear(1)` |
| Loss | BCEWithLogitsLoss |
| Opt | Adam lr=2e-4, β=(0.5, 0.999) |

### CGAN — Fashion MNIST (PRO)
| Блок | Детали |
|------|--------|
| LabelEncoder | CNN (3×Conv2d) → Linear(256) + LayerNorm |
| G | `Linear(z+emb→512·7·7)` → `ConvT×2` → `Conv×2` + Tanh |
| D | Ветка изображения \|\| Ветка лейбла → concat → `Conv×2` → `Linear(1)` |
| Loss | 3-way: real+OK / fake+OK / real+wrong |
| Opt | Adam lr=1e-4 + CosineAnnealingLR |

### DCGAN — Cats vs Dogs (Kaggle)
| Блок | Детали |
|------|--------|
| G | `z(128,1,1)` → `ConvT(512,4×4)` → `ConvT(256)` → `ConvT(128)` → `ConvT(64)` → `ConvT(3)` + Tanh — выход 64×64 RGB |
| D | `Conv(64)` → `Conv(128)` → `Conv(256)` → `Conv(512)` → `Conv(1)` |
| Трюки | Label smoothing (0.85–1.0), 2 шага G на 1 шаг D, lr_D × 0.4 |
| Opt | Adam lr=2e-4/8e-5, β=(0.5, 0.999) |

**Ultra PRO**: интерактивный canvas (HTML5 + JS) → PIL resize 28×28 → тот же LabelEncoder → генерация + интерполяция в embedding-пространстве.